In [5]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
import smarts_cerebellum.globals as gl

atlas = 'Nettekoven_2024'
segment = 'GM'

# observed means
obs = pd.read_csv(os.path.join(gl.baseDir, f'MNISymC_{segment}', f'summary_MNISymC_{atlas}_{segment}_mod.tsv'), sep = '\t')
obs = obs[obs.isPatient == 1]
obs['region_bilat'] = obs.regionname.str[:-1]
obs['hemisphere'] = obs.regionname.str[-1]

other_rois = ['M1', 'M2', 'M4', 'A1', 'A2', 'A3', 'D1', 'D2', 'D3', 'D4', 'S1', 'S2', 'S3', 'S4', 'S5']
obs['is_M3'] = np.where(obs.region_bilat == 'M3', 'M3', 'other')
bilat = (obs.groupby(['subj_id', 'hemisphere', 'Week', 'is_M3'], as_index = False))['mean'].mean()
df = (bilat.groupby(['subj_id', 'is_M3', 'hemisphere'], as_index = False))['mean'].mean()

In [6]:
# 2x2 (repeated measures) ANOVA
import pingouin as pg
aov = pg.rm_anova(data = df,
                  dv = 'mean',
                  within = ['is_M3', 'hemisphere'],
                  subject = 'subj_id')
aov

,Source,SS,ddof1,ddof2,MS,F,p_unc,p_GG_corr,ng2,eps
0,is_M3,0.003290,1,38,0.003290,1.773257,0.190913,0.190913,0.003707,1.0
1,hemisphere,0.010013,1,38,0.010013,10.281182,0.002725,0.002725,0.011195,1.0
2,is_M3 * hemisphere,0.002056,1,38,0.002056,2.523131,0.120475,0.120475,0.002319,1.0


The problem here is that region sizes aren't uniform, so 'regionname' effect is also capturing difference in region sizes (??). We should normalize for that, right?

In [13]:
# 2x3 (repeated measures) ANOVA
import pingouin as pg
aov = pg.rm_anova(data = obs[obs.region_bilat.isin(['M3', 'M4', 'S1'])],
                  dv = 'mean',
                  within = ['region_bilat', 'hemisphere'],
                  subject = 'subj_id')
aov

,Source,SS,ddof1,ddof2,MS,F,p_unc,p_GG_corr,ng2,eps
0,region_bilat,0.011925,2,76,0.005963,1.513244,0.226754,0.227655,0.006890,0.946968
1,hemisphere,0.005947,1,38,0.005947,3.778604,0.059344,0.059344,0.003448,1.000000
2,region_bilat * hemisphere,0.008083,2,76,0.004042,2.969571,0.057312,0.061424,0.004680,0.930375
